<a href="https://colab.research.google.com/github/lkodwani27/Enterprise-PDF-RAG-Chatbot/blob/main/PDF_RAG_LongContext_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## Objectives

- Handle **large PDFs** safely using long-context models
- Implement enterprise-grade text chunking and overlap
- Visualize and validate chunks
- Build an in-memory FAISS vector store
- Perform semantic retrieval using Sentence Transformers
- Use a **long-context instruction-tuned generator**
- Enable chatbot-style interaction grounded strictly in PDF content
- Demonstrate hallucination mitigation at scale



## Step 1: Environment Setup

We rely entirely on open-source libraries. Long-T5 is chosen for its extended context window and instruction-following behavior, making it significantly more stable than GPT-2 for document-grounded tasks.


In [2]:
!pip install -q transformers sentence-transformers faiss-cpu pypdf torch accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 19.4 MB/s eta 0:00:00



## Step 2: Upload a Large PDF

Upload any large PDF (hundreds of pages supported). This simulates enterprise document ingestion pipelines.


In [3]:
## PDF Downlaod link : https://irdai.gov.in/documents/37343/993134/Health+Companion-Health+Insurance+Plan_GEN617.pdf/90da4af9-8b4b-d775-a71c-8c4c94d45050?version=1.1&t=1668330173300&download=true

In [4]:
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f"Uploaded PDF: {pdf_path}")
# print(f"Type of 'uploaded:'{type(uploaded)}")
# print(f"Numer of items: {len(uploaded)}")
# print(f"Keys: {list(uploaded.keys())}")

Saving Health Companion-Health Insurance Plan_GEN617.pdf to Health Companion-Health Insurance Plan_GEN617.pdf
Uploaded PDF: Health Companion-Health Insurance Plan_GEN617.pdf



## Step 3: Extract Text from PDF

We extract text page-by-page to preserve document flow while avoiding memory spikes.


In [5]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)
raw_text = ""

for i, page in enumerate(reader.pages):
  text = page.extract_text()
  if text:
    raw_text += text + "\n"


print(f"Extracted character: {len(raw_text)}")
print(raw_text[:1000])



Extracted character: 53092
Health Companion – Health Insurance Plan  POLICY DOCUMENT 
 
 
Page 1 of 25 
Policy Document – Part II 
1. Terms & Conditions 
The insurance cover provided under this Policy to the Insured Person up to the Sum Insured is and shall 
be subject to (a) the terms and conditions of this Policy and (b) the receipt of premium, and (c) the 
information You provided to Us (including by way of the Proposal or Information Summary Sheet) on Your 
behalf and on behalf of all persons to be insured. Please inform Us immediately of any change in the 
address, nature of job, state of health, or of any other changes affecting You or any Insured Person.  
2. Benefits 
The Policy covers reasonable expenses incurred towards medical treatment taken during the Policy 
Period for an Illness, Accident or condition described below if this is contracted or sustained by an 
Insured Person during the Policy Period and subj ect always to the Sum Insured, any subsidiary limit 
specified in


## Step 4: Chunking Strategy for Large Documents

Chunking is critical even with long-context models. We use moderately large chunks with overlap to balance retrieval accuracy and generation stability.


In [6]:
def chunk_text(text, chunk_size = 500, overlap=100):
  words = text.split()
  chunks = []
  for i in range(0, len(words), chunk_size - overlap):
    chunk = " ".join(words[i:i + chunk_size])
    chunks.append(chunk)
  return chunks

chunks = chunk_text(raw_text)

print(f"Total No of Chunks : {len(chunks)}")

Total No of Chunks : 21


##Step 5: Inspect Sample Chunks


Inspecting chunks is mandatory for debugging and governance in enterprise RAG systems.


In [7]:
for i, chunk in enumerate(chunks[:5]):
  print(f"----Chunks {i+1}---")
  print(chunk[:800])
  print()

----Chunks 1---
Health Companion – Health Insurance Plan POLICY DOCUMENT Page 1 of 25 Policy Document – Part II 1. Terms & Conditions The insurance cover provided under this Policy to the Insured Person up to the Sum Insured is and shall be subject to (a) the terms and conditions of this Policy and (b) the receipt of premium, and (c) the information You provided to Us (including by way of the Proposal or Information Summary Sheet) on Your behalf and on behalf of all persons to be insured. Please inform Us immediately of any change in the address, nature of job, state of health, or of any other changes affecting You or any Insured Person. 2. Benefits The Policy covers reasonable expenses incurred towards medical treatment taken during the Policy Period for an Illness, Accident or condition described below 

----Chunks 2---
(including Chemotherapy, Radiotherapy, Hemodialysis, any procedure which needs a period of specialized observation or care after completion of the procedure) where su

##Step 6: Embedding Generation (Retriever Encoder)

We use Sentence Transformers optimized for semantic similarity. Retrieval quality directly depends on this choice.

In [8]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(chunks, show_progress_bar=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

## Inspect the Embeddings



In [9]:
import numpy as np

print(f"Embedding Inspection ")
print(f"=" * 50)
print(f"Shape = {embeddings.shape}")
print(f"DataType: {embeddings.dtype}")
print(f"First Chunk's embedding (first 10 values)")
print(embeddings[0][:10])
first_emb = embeddings[0]
print(f"Min Value : {first_emb.min():.4f}")
print(f"Max Value : {first_emb.max():.4f}")
print(f"L2 Norm  : {np.linalg.norm(first_emb):.4f}")


Embedding Inspection 
Shape = (21, 384)
DataType: float32
First Chunk's embedding (first 10 values)
[-0.04942761  0.1310487   0.06123729 -0.00595545  0.08767772  0.08155014
  0.07016151  0.10287652 -0.02240624 -0.00500618]
Min Value : -0.1578
Max Value : 0.1362
L2 Norm  : 1.0000


##Step 7: FAISS(Facebook AI Similarity Serach) Vector Store Construction

FAISS enables fast semantic search even for large chunk collections.

In [10]:
import faiss
import numpy as np


dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print(f"Total vectors in index: {index.ntotal}")
print("FAISS index ready")


Total vectors in index: 21
FAISS index ready


In [ ]:
def retrieve_context(query, top_k=5):
  query_embedding = embedder.encode([query])
  distance, indices = index.search(np.array(query_embedding), top_k)
  return [chunks[i] for i in indices[0]]

##Step 9: Load Long-Context Generator (Long-T5)

Long-T5 supports large input contexts and is instruction-tuned, making it ideal for RAG.

In [16]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/long-t5-tglobal-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model

Loading weights:   0%|          | 0/297 [00:00<?, ?it/s]

LongT5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): LongT5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): LongT5Block(
        (layer): ModuleList(
          (0): LongT5LayerTransientGlobalSelfAttention(
            (TransientGlobalSelfAttention): LongT5TransientGlobalAttention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
              (global_relative_attention_bias): Embedding(32, 12)
              (global_input_layer_norm): LongT5LayerNorm()
            )
            (layer_norm): LongT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): LongT5LayerFF(
            (DenseReluDense

In [14]:

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/long-t5-tglobal-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model


pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/297 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LongT5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): LongT5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): LongT5Block(
        (layer): ModuleList(
          (0): LongT5LayerTransientGlobalSelfAttention(
            (TransientGlobalSelfAttention): LongT5TransientGlobalAttention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
              (global_relative_attention_bias): Embedding(32, 12)
              (global_input_layer_norm): LongT5LayerNorm()
            )
            (layer_norm): LongT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): LongT5LayerFF(
            (DenseReluDense